# ShelfWise — Gemma 4 fine-tune (LoRA, local, on this GPU)

**Run every cell top to bottom (Kernel -> Restart & Run All).** This trains a real LoRA adapter on
top of `google/gemma-4-12B-it` using the synthetic SFT data `shelfwise_synthdata` already generates,
then serves it locally via `vllm` so ShelfWise's routine agents (inventory, expiry, demand,
opportunity, simulation) can call it at zero token cost — while critic/executive keep escalating to
Fireworks (MiniMax/Kimi K). This also qualifies the project for the hackathon's "Best Use of Gemma"
bonus challenge.

**What this notebook proves, section by section:**

1. Generates a real-scale synthetic SFT dataset (not the 64-row smoke test) and writes it to
   `data/training/`.
2. Loads `google/gemma-4-12B-it` in bf16 and wraps it with a LoRA adapter (base weights frozen,
   only a small adapter trains — this is what fits comfortably in this GPU's ~48GB budget).
3. Builds a properly masked training set: only the completion tokens contribute to loss, not the
   prompt — a naive implementation that trains on the whole sequence wastes signal and can hurt
   quality.
4. Trains for a **wall-clock time budget**, not a fixed step count, so it fits inside whatever
   compute window you actually have left — and checkpoints to `/workspace/checkpoints/` (persistent
   storage) periodically, so a quota reset or pod restart mid-run does not lose progress; it resumes
   automatically from the latest checkpoint.
5. Saves the final adapter and runs one generation smoke test before you trust it.
6. Prints the exact `vllm serve` command to serve the fine-tuned adapter locally, and the exact
   `LLM_BASE_URL` to point the running ShelfWise backend at it.

**Before running:** this notebook was built and its training mechanics (masking, LoRA wrapping,
checkpoint save/resume, adapter save/load, generation) were verified end-to-end against a tiny
placeholder model on a machine with no GPU and no ROCm — there was no way to download or run the
real 12B model there. The Gemma-specific parts (the actual download, memory fit, and vLLM's support
for serving this specific architecture's LoRA adapter) can only be confirmed by running this for
real, here, on this box. If something in those specific parts errors, that is expected to be
possible and worth reporting back rather than a sign the whole approach is wrong.

Set these environment variables before starting the kernel to control scope:

- `SFT_ROWS` (default `3000`) — how many synthetic training rows to generate.
- `MAX_TRAIN_HOURS` (default `3.5`) — wall-clock budget for the training cell, leaving headroom in
  a 5-hour window for setup, serving, and verification.
- `CHECKPOINT_DIR` (default `/workspace/checkpoints/shelfwise-gemma`) — where checkpoints and the
  final adapter are written; must be under `/workspace` to survive a pod restart.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

MODEL_ID = "google/gemma-4-12B-it"
SFT_ROWS = int(os.environ.get("SFT_ROWS", "3000"))
MAX_TRAIN_HOURS = float(os.environ.get("MAX_TRAIN_HOURS", "3.5"))
CHECKPOINT_DIR = Path(os.environ.get("CHECKPOINT_DIR", "/workspace/checkpoints/shelfwise-gemma"))
ADAPTER_DIR = CHECKPOINT_DIR / "final_adapter"
SFT_PATH = ROOT / "data" / "training" / "shelfwise_sft_gemma.jsonl"

print(f"repo root: {ROOT}")
print(f"model: {MODEL_ID}")
print(f"sft rows: {SFT_ROWS}")
print(f"max train hours: {MAX_TRAIN_HOURS}")
print(f"checkpoint dir: {CHECKPOINT_DIR}")

### Check torch actually sees this GPU before doing anything else

This box's GPU reports as `gfx1100` (RDNA3/Navi 31 - an AMD Radeon PRO W7900-class workstation card, not an Instinct MI300X; different architecture family, but still fully AMD ROCm compute). A plain `pip install torch` grabs the default CUDA build, which will not see this GPU at all - it needs the ROCm build, matched to this box's actual ROCm version. Fail loudly here with the exact fix, rather than a confusing `ModuleNotFoundError` or a silent CPU fallback deep inside the training cell later.

In [ ]:
import subprocess


def _detect_rocm_version() -> str | None:
    version_file = Path("/opt/rocm/.info/version")
    if version_file.exists():
        text = version_file.read_text(encoding="utf-8").strip()
        return ".".join(text.split(".")[:2])
    try:
        result = subprocess.run(["rocminfo"], capture_output=True, text=True)
    except OSError:
        return None
    for line in result.stdout.splitlines():
        if "HSA Runtime Version" in line:
            return line.split(":")[-1].strip().rsplit(".", 1)[0]
    return None


def _check_torch_gpu() -> None:
    rocm_version = _detect_rocm_version()
    if rocm_version:
        wheel_index = f"https://download.pytorch.org/whl/rocm{rocm_version}"
    else:
        wheel_index = "https://download.pytorch.org/whl/rocm6.2"
        print(
            "could not auto-detect this box's ROCm version from rocminfo or "
            "/opt/rocm/.info/version - defaulting to rocm6.2 in the command below; check "
            "`rocminfo` yourself and adjust the URL if this box runs a different version."
        )
    try:
        import torch
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "torch is not installed in this environment. Run this in a terminal, then restart "
            f"the kernel:\n\n  pip install torch --index-url {wheel_index}\n"
        ) from exc

    if not torch.cuda.is_available():
        raise RuntimeError(
            "torch is installed but torch.cuda.is_available() is False - ROCm reuses the CUDA "
            "API surface, so this means torch is not linked against ROCm correctly for this "
            "card. Try, in order: (1) reinstalling with the ROCm wheel index matching this box - "
            f"pip install torch --index-url {wheel_index} --force-reinstall, then restart the "
            "kernel; (2) if that still fails, set HSA_OVERRIDE_GFX_VERSION=11.0.0 (this card's "
            "gfx1100 arch family) before starting the kernel."
        )

    props = torch.cuda.get_device_properties(0)
    print(f"torch {torch.__version__}, ROCm/HIP: {getattr(torch.version, 'hip', None)}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM: {props.total_memory / 1024**3:.1f} GB, "
        f"arch: {getattr(props, 'gcnArchName', 'unknown')}"
    )


_check_torch_gpu()

### Install training dependencies

Skips `bitsandbytes`/4-bit quantization on purpose: ROCm support for it is new enough (2026) that it
adds real setup risk for a hackathon deadline, and a bf16 LoRA fine-tune of a 12B model already fits
comfortably in this GPU's ~48GB — base weights frozen (~24GB in bf16) plus a small trainable adapter,
with no optimizer-state memory for the (frozen) base model.

In [ ]:
install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.45",
        "peft>=0.13",
        "datasets>=3.0",
        "accelerate>=1.0",
    ],
    capture_output=True,
    text=True,
)
print(install.stdout[-1500:])
if install.returncode != 0:
    print(install.stderr[-1500:])
assert install.returncode == 0, "dependency install failed"

## 1. Generate the synthetic SFT dataset

Uses the same `shelfwise_synthdata.generate_agent_sft` generator the rest of this repo already
relies on, just at real scale instead of the 64-row smoke default, and writes it with the same
`shelfwise_mlops.export_sft_jsonl` this repo already uses for `data/training/`.

In [ ]:
from shelfwise_mlops import export_sft_jsonl
from shelfwise_synthdata import generate_agent_sft

SFT_SEED = 20260709
row_count = export_sft_jsonl(generate_agent_sft(SFT_SEED, n=SFT_ROWS), SFT_PATH)
print(f"wrote {row_count} SFT rows to {SFT_PATH}")
assert row_count == SFT_ROWS

## 2. Load the base model and wrap it with LoRA

`target_modules="all-linear"` asks PEFT to find every linear layer in whatever architecture this
model actually is, rather than hand-naming layer names for a model family (Gemma 4) that is new
enough that guessing wrong is a real risk. Only the adapter trains; the base model stays frozen.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
assert tokenizer.chat_template is not None, (
    f"{MODEL_ID} did not ship a chat template. Do not substitute a hand-written one here - "
    "an instruction-tuned model's real template defines the exact format it was trained to "
    "expect, and silently guessing a different one would train on a mismatched format with "
    "no visible warning. Check the model repo's tokenizer_config.json instead."
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    target_modules="all-linear",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Build a properly masked training set

Each row becomes one training example: the chat-formatted prompt (masked out of the loss, label
`-100`) followed by the target completion (the `output` evidence dict, as JSON — the same shape the
real agents already produce) and an end-of-sequence token. Only the completion tokens contribute to
the loss; the model is not asked to predict the prompt it was given.

In [ ]:
from datasets import Dataset


def _read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            rows.append(json.loads(line))
    return rows


def build_examples(rows: list[dict]) -> list[dict]:
    examples = []
    for row in rows:
        prompt_text = tokenizer.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=True
        )
        completion_text = json.dumps(row["output"], sort_keys=True) + tokenizer.eos_token

        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        completion_ids = tokenizer(completion_text, add_special_tokens=False)["input_ids"]

        input_ids = prompt_ids + completion_ids
        labels = [-100] * len(prompt_ids) + completion_ids
        examples.append({"input_ids": input_ids, "labels": labels})
    return examples


rows = _read_jsonl(SFT_PATH)
examples = build_examples(rows)
for example in examples[:5]:
    assert any(label != -100 for label in example["labels"]), "completion must be unmasked"
    assert example["labels"][0] == -100, "prompt tokens must be masked"

train_dataset = Dataset.from_list(examples)
lengths = [len(item["input_ids"]) for item in examples[:5]]
print(f"dataset ready: {len(train_dataset)} examples, sample token lengths {lengths}")


def collate(batch: list[dict]) -> dict:
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, labels, attention_mask = [], [], []
    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        labels.append(item["labels"] + [-100] * pad_len)
        attention_mask.append([1] * len(item["input_ids"]) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }

## 4. Train — bounded by wall-clock time, checkpointed, auto-resuming

`TimeBudgetCallback` stops training once `MAX_TRAIN_HOURS` elapses, whatever step that happens to
be — this run does not need to guess the right epoch count in advance. `save_strategy="steps"`
writes checkpoints to `CHECKPOINT_DIR` (under `/workspace`, so they survive a pod restart per the
hackathon FAQ's own checkpointing guidance); if a checkpoint from an earlier, interrupted run already
exists, this resumes from it instead of starting over.

In [ ]:
from transformers import Trainer, TrainerCallback, TrainingArguments


class TimeBudgetCallback(TrainerCallback):
    def __init__(self, max_seconds: float) -> None:
        self.deadline = time.monotonic() + max_seconds

    def on_step_end(self, args, state, control, **kwargs):
        if time.monotonic() >= self.deadline:
            control.should_training_stop = True
        return control


training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    report_to=[],
)

existing_checkpoints = (
    sorted(CHECKPOINT_DIR.glob("checkpoint-*")) if CHECKPOINT_DIR.exists() else []
)
resume_from = str(existing_checkpoints[-1]) if existing_checkpoints else None
if resume_from:
    print(f"resuming from existing checkpoint: {resume_from}")
else:
    print("no existing checkpoint found - starting fresh")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate,
    callbacks=[TimeBudgetCallback(max_seconds=MAX_TRAIN_HOURS * 3600)],
)

train_result = trainer.train(resume_from_checkpoint=resume_from)
print(json.dumps(train_result.metrics, indent=2))

## 5. Save the final adapter

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
saved_files = sorted(p.name for p in ADAPTER_DIR.glob("*"))
print(f"final adapter saved to {ADAPTER_DIR}")
print(f"files: {saved_files}")
assert any("adapter_model" in name for name in saved_files), "adapter weights file missing"

## 6. Generation smoke test

Loads the saved adapter fresh (not the in-memory training object) and generates one completion for
a sample ShelfWise agent prompt, so you have direct evidence of what it actually produces before
trusting it in the running app.

In [ ]:
from peft import PeftModel

base_for_generation = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
loaded = PeftModel.from_pretrained(base_for_generation, str(ADAPTER_DIR))

sample_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Assess expiry risk for synthetic SKU 4011."}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(sample_prompt, return_tensors="pt").to(loaded.device)
generated = loaded.generate(**inputs, max_new_tokens=120, do_sample=False)
output_text = tokenizer.decode(
    generated[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
)
print("sample completion:")
print(output_text)

## 7. Serve it and wire it into ShelfWise

`vllm serve` needs a terminal, not a notebook cell that blocks forever — this prints the exact
command to run in a terminal tab in this same Jupyter pod, plus the exact environment variable to
point the running ShelfWise backend at it so the routine agents (inventory, expiry, demand,
opportunity, simulation) call this local adapter at zero token cost, while critic/executive keep
escalating to Fireworks.

In [ ]:
print("Run this in a terminal tab in this same Jupyter pod:\n")
print(
    f"vllm serve {MODEL_ID} --enable-lora "
    f"--lora-modules shelfwise={ADAPTER_DIR} "
    f"--port 8000 --gpu-memory-utilization 0.6\n"
)
print(
    "Then point the ShelfWise backend at it (routine agents only; "
    "critic/executive stay on Fireworks):\n"
)
print("  LLM_BASE_URL=http://127.0.0.1:8000")
print("  LLM_ROUTINE_MODEL=shelfwise")